# Muraqam (مُرقّم) — AraBERTv02 **base + large**, 5-fold CV, **2×T4 in parallel**

Same pipeline as before (metric-identical word-gap tokenizer, multi-label sigmoid head, focal loss,
sliding windows, 5-fold CV, per-class threshold tuning, everything persisted to `SAVE_DIR`) —
but the 3-encoder ensemble is replaced by exactly two models:

| key | model | GPU |
|---|---|---|
| `base`  | `aubmindlab/bert-base-arabertv02`  | 0 |
| `large` | `aubmindlab/bert-large-arabertv02` | 1 |

**Why v02 and not v2:** `v2` is the *Farasa-segmented* variant — it expects text pre-segmented into
morphemes, which changes the whitespace word count and would silently break the host metric's
alignment unit. `v02` is trained on unsegmented text, so raw words go in as-is. Do **not** run
`ArabertPreprocessor` here for the same reason (it re-spaces punctuation and would shift word counts).

**Parallelism:** the whole training loop is dumped to `worker.py`; the launcher fires two
`nohup` processes, one pinned to each T4 via `CUDA_VISIBLE_DEVICES`, and a monitor cell tails both
logs. Wall-clock ≈ the *large* run alone instead of base + large serially. Each worker also does its
own fold-bagged **test inference on its own GPU**, so the blend/submission cells need no GPU.

**Everything still gets saved** to `SAVE_DIR`: fold checkpoints (`*_fold{k}.pt`), full OOF probs
(`oof_{slug}.pkl`), bagged test probs (`test_probs_{slug}.pkl`), fold map, tuned thresholds +
blend weight (`blend.json`). Set `REUSE_SAVED=True` and reruns skip any fold already on disk.

> Requires **Accelerator = GPU T4 ×2** in the notebook settings.

## 0. Config (single source of truth — written to `config.json` for both workers)

In [ ]:
# =========================================================================
#  Config. Both worker processes read this JSON, so the notebook and the
#  workers can never drift apart. Edit ONLY here.
# =========================================================================
!pip install -q transformers

import os, json, torch

TRAIN_CSV = "/kaggle/input/competitions/muraqqamchallenge/train.csv"
TEST_CSV  = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
if not os.path.exists(TRAIN_CSV):
    TRAIN_CSV = "train.csv"; TEST_CSV = "test.csv"

SAVE_DIR = "/kaggle/working/muraqam_cv"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)

CFG = dict(
    SEED       = 2026,
    TRAIN_CSV  = TRAIN_CSV,
    TEST_CSV   = TEST_CSV,
    SAVE_DIR   = SAVE_DIR,
    MARKS      = ['.', '،', '؟', '!', ':', '؛', '-'],   # canonical order, do not touch

    # ---- CV ----
    N_FOLDS    = 5,
    GROUP_COL  = None,        # e.g. "book" -> GroupKFold; None -> doc-level KFold
    REUSE_SAVED= True,        # skip folds whose checkpoint already exists

    # ---- windows ----
    CHUNK_WORDS= 180,
    STRIDE     = 140,         # 40-word overlap
    MAX_LEN    = 448,

    # ---- shared training knobs ----
    FOCAL_GAMMA = 2.0,
    HEAD_DROPOUT= 0.1,
    WEIGHT_DECAY= 0.01,
    WARMUP_FRAC = 0.10,
    SCHEDULE    = "onecycle", # "onecycle" | "cosine" | "linear"
    USE_AMP     = True,

    # ---- rules ----
    HONORIFICS  = ["ﷺ"],
    FORCE_HONORIFIC_RULE = False,

    # ---- threshold search bounds (exact sweep inside these bounds) ----
    THR_LO = 0.05, THR_HI = 0.90,

    # ---- THE TWO MODELS: one per T4 ----
    MODELS = {
        "base": dict(
            name = "aubmindlab/bert-base-arabertv02",
            gpu  = 0,
            hp   = dict(lr=2e-5, batch=8, grad_accum=1, max_epochs=30, patience=4,
                        use_llrd=False, llrd_decay=0.95, grad_ckpt=False),
        ),
        "large": dict(
            name = "aubmindlab/bert-large-arabertv02",
            gpu  = 1,
            # large @ 448 tok on a 16GB T4: batch 4 x accum 2 = same effective batch as base.
            # lower LR + LLRD because large is the one that actually diverges under fp16.
            hp   = dict(lr=1e-5, batch=4, grad_accum=2, max_epochs=30, patience=4,
                        use_llrd=True, llrd_decay=0.95, grad_ckpt=False),
        ),
    },
)

with open("/kaggle/working/config.json", "w") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)

MARKS = CFG["MARKS"]; NUM_MARKS = len(MARKS)
N_FOLDS = CFG["N_FOLDS"]

n_gpu = torch.cuda.device_count()
print("GPUs visible:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
if n_gpu < 2:
    print("!! Only one GPU -> set Accelerator = 'GPU T4 x2'. "
          "The launcher will still work but both workers would share one card.")
print("config.json written | train:", TRAIN_CSV)

## 1. Host metric (verbatim)

In [ ]:
# Host metric (verbatim) — for trustworthy LOCAL validation before you submit.
class ParticipantVisibleError(Exception):
    pass

"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 2. Shared library → `muraqam_lib.py`

Cells 2–5 of the old notebook (tokenizer/labels, dataset, model+focal, train/infer, thresholds)
now live in one importable file, so the two worker processes and the notebook run *identical* code.
Two functional upgrades over the old cells:

* **gradient accumulation** (so `large` can use batch 4 × 2 on a T4 with the same effective batch as `base`);
* **exact threshold search** — instead of a 52-point grid, every candidate cutoff is swept in
  `O(n log n)` via `F1 = 2·TP/(k + n_pos)` on the prob-sorted column. Strictly better for `؛` and `!`,
  and cheap enough to re-run inside the blend-weight loop.

In [ ]:
%%writefile /kaggle/working/muraqam_lib.py
# =========================================================================
#  muraqam_lib.py  —  shared code for BOTH GPU workers and the notebook.
#  Everything the workers need (tokenizer/labels/dataset/model/train/infer)
#  lives here so the two processes are byte-identical apart from the model.
# =========================================================================
import os, json, math, random, pickle, gc
from functools import partial

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

CFG = json.load(open("/kaggle/working/config.json"))

SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

MARKS      = CFG["MARKS"]
M2I        = {m: i for i, m in enumerate(MARKS)}
NUM_MARKS  = len(MARKS)
VALID_SYMBOLS = set(''.join(MARKS))

TRAIN_CSV  = CFG["TRAIN_CSV"]; TEST_CSV = CFG["TEST_CSV"]
SAVE_DIR   = CFG["SAVE_DIR"];  os.makedirs(SAVE_DIR, exist_ok=True)
N_FOLDS    = CFG["N_FOLDS"];   GROUP_COL = CFG["GROUP_COL"]
REUSE_SAVED = CFG["REUSE_SAVED"]

CHUNK_WORDS = CFG["CHUNK_WORDS"]; STRIDE = CFG["STRIDE"]; MAX_LEN = CFG["MAX_LEN"]
FOCAL_GAMMA = CFG["FOCAL_GAMMA"]; HEAD_DROPOUT = CFG["HEAD_DROPOUT"]
WEIGHT_DECAY = CFG["WEIGHT_DECAY"]; WARMUP_FRAC = CFG["WARMUP_FRAC"]
SCHEDULE = CFG["SCHEDULE"]; USE_AMP = CFG["USE_AMP"]
HONORIFICS = set(CFG["HONORIFICS"]); FORCE_HONORIFIC_RULE = CFG["FORCE_HONORIFIC_RULE"]
THR_LO, THR_HI = CFG["THR_LO"], CFG["THR_HI"]

# =========================================================================
#  1. Word-gap tokenizer (IDENTICAL to the host metric) + labels + rules
# =========================================================================
def tokenize_gold(text):
    """-> (leading_gap, [(word, trailing_gap), ...]).  Matches host metric."""
    leading=[]; pairs=[]; cur=[]; in_word=False
    for ch in str(text):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for w,g in pairs]

def gold_to_labels(final_text):
    """(words, Y[n_words, NUM_MARKS]) multi-hot of the gap AFTER each word."""
    _, pairs = tokenize_gold(final_text)
    words=[w for w,_ in pairs]
    Y=np.zeros((len(words), NUM_MARKS), dtype=np.float32)
    for i,(_,gap) in enumerate(pairs):
        for c in gap:
            if c in M2I: Y[i, M2I[c]] = 1.0
    return words, Y

def words_of_raw(raw):
    """Word units exactly as the metric derives them: whitespace split."""
    return str(raw).strip().split()

MARK_ORDER = MARKS

def reconstruct(words, pred_multi):
    """words + per-word multi-hot -> final_text (word count preserved)."""
    toks=[]
    for w, row in zip(words, pred_multi):
        marks=''.join(m for m in MARK_ORDER if row[M2I[m]]>0)
        toks.append(w+marks)
    return ' '.join(toks)

def apply_honorific_rule(words, pred_multi):
    if not FORCE_HONORIFIC_RULE: return pred_multi
    P=pred_multi.copy(); di=M2I['-']
    for i,w in enumerate(words):
        if w in HONORIFICS:
            P[i, di]=1.0
            if i>0: P[i-1, di]=1.0
    return P

def probs_to_multihot(words, P, thresholds):
    pred=(P>=np.asarray(thresholds)[None,:]).astype(np.float32)
    return apply_honorific_rule(words, pred)

# =========================================================================
#  2. Data + folds  (deterministic -> both workers get the SAME split)
# =========================================================================
def load_df(verbose=True):
    from sklearn.model_selection import KFold, GroupKFold
    df = pd.read_csv(TRAIN_CSV).dropna(subset=["text","final_text"]).reset_index(drop=True)
    fold_of = np.full(len(df), -1, dtype=int)
    if GROUP_COL is not None and GROUP_COL in df.columns:
        gkf = GroupKFold(n_splits=N_FOLDS)
        for k,(_,va) in enumerate(gkf.split(np.arange(len(df)), groups=df[GROUP_COL].values)):
            fold_of[va]=k
        if verbose: print(f"GroupKFold by '{GROUP_COL}' -> {N_FOLDS} folds")
    else:
        kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        for k,(_,va) in enumerate(kf.split(np.arange(len(df)))):
            fold_of[va]=k
        if verbose: print(f"KFold (doc-level, shuffled, seed={SEED}) -> {N_FOLDS} folds")
    df["fold"]=fold_of
    return df

# =========================================================================
#  3. Sliding-window dataset (predict at LAST subword of each word)
# =========================================================================
def chunk_words(words, Y=None):
    n=len(words)
    if n<=CHUNK_WORDS:
        yield words, (Y if Y is not None else None), 0; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS, n)
        yield words[s:e], (Y[s:e] if Y is not None else None), s
        if e==n: break
        s+=STRIDE

class PunctDataset(Dataset):
    def __init__(self, rows, tokenizer, has_labels=True):
        self.samples=[]; self.tok=tokenizer; self.has_labels=has_labels
        for r in rows:
            words=words_of_raw(r["text"])
            Y = gold_to_labels(r["final_text"])[1] if has_labels else None
            for wslice,Yslice,start in chunk_words(words,Y):
                self.samples.append((wslice,Yslice))
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        words,Y=self.samples[i]
        enc=self.tok(words, is_split_into_words=True, truncation=True,
                     max_length=MAX_LEN, return_tensors=None)
        word_ids=enc.word_ids()
        last_pos={}
        for pos,wid in enumerate(word_ids):
            if wid is not None: last_pos[wid]=pos
        active=np.zeros(len(word_ids),dtype=bool)
        labels=np.zeros((len(word_ids),NUM_MARKS),dtype=np.float32)
        wid_at=np.full(len(word_ids),-1,dtype=np.int64)
        for wid,pos in last_pos.items():
            active[pos]=True; wid_at[pos]=wid
            if Y is not None and wid<len(Y): labels[pos]=Y[wid]
        return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],
                "active":active,"labels":labels,"word_ids":wid_at}

def collate(batch, pad_id):
    maxlen=max(len(b["input_ids"]) for b in batch); B=len(batch)
    input_ids=np.full((B,maxlen),pad_id,dtype=np.int64)
    attn=np.zeros((B,maxlen),dtype=np.int64)
    active=np.zeros((B,maxlen),dtype=bool)
    labels=np.zeros((B,maxlen,NUM_MARKS),dtype=np.float32)
    wids=np.full((B,maxlen),-1,dtype=np.int64)
    for i,b in enumerate(batch):
        L=len(b["input_ids"])
        input_ids[i,:L]=b["input_ids"]; attn[i,:L]=b["attention_mask"]
        active[i,:L]=b["active"]; labels[i,:L]=b["labels"]; wids[i,:L]=b["word_ids"]
    return (torch.tensor(input_ids),torch.tensor(attn),torch.tensor(active),
            torch.tensor(labels),torch.tensor(wids))

# =========================================================================
#  4. Multi-label token classifier + focal loss
# =========================================================================
class PunctModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.drop=nn.Dropout(HEAD_DROPOUT)
        self.head=nn.Linear(h,NUM_MARKS)
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(out))

def focal_bce(logits, targets, active, gamma=FOCAL_GAMMA, pos_weight=None):
    logits=logits[active]; targets=targets[active]
    if logits.numel()==0: return logits.sum()*0.0
    bce=nn.functional.binary_cross_entropy_with_logits(
        logits,targets,reduction='none',pos_weight=pos_weight)
    p=torch.sigmoid(logits); p_t=p*targets+(1-p)*(1-targets)
    return (((1-p_t)**gamma)*bce).mean()

def compute_pos_weight(rows):
    pos=np.zeros(NUM_MARKS); tot=0
    for r in rows:
        _,Y=gold_to_labels(r["final_text"]); pos+=Y.sum(0); tot+=len(Y)
    neg=tot-pos
    w=np.clip(neg/np.clip(pos,1,None),1.0,20.0)
    return torch.tensor(w,dtype=torch.float32)

# =========================================================================
#  5. Optimizer / scheduler / train / infer
# =========================================================================
def build_optimizer(model, base_lr, weight_decay, use_llrd=False, llrd_decay=0.95):
    if not use_llrd:
        return torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    try:
        layers = model.backbone.encoder.layer; n=len(layers)
    except Exception:
        return torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=weight_decay)
    groups=[]
    head_params=[p for nme,p in model.named_parameters()
                 if not nme.startswith("backbone.encoder.layer") and not nme.startswith("backbone.embeddings")]
    groups.append({"params":head_params,"lr":base_lr,"weight_decay":weight_decay})
    for i,layer in enumerate(layers):
        groups.append({"params":list(layer.parameters()),
                       "lr":base_lr*(llrd_decay**(n-1-i)),"weight_decay":weight_decay})
    try:
        groups.append({"params":list(model.backbone.embeddings.parameters()),
                       "lr":base_lr*(llrd_decay**n),"weight_decay":weight_decay})
    except Exception: pass
    return torch.optim.AdamW(groups)

def build_scheduler(opt, total_steps, warmup_frac, base_lr):
    warm=int(total_steps*warmup_frac)
    if SCHEDULE=="onecycle":
        lrs=[g["lr"] for g in opt.param_groups]
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lrs, total_steps=total_steps,
                                                   pct_start=warmup_frac)
    from torch.optim.lr_scheduler import LambdaLR
    def lr_lambda(step):
        if step<warm: return step/max(1,warm)
        prog=(step-warm)/max(1,total_steps-warm)
        if SCHEDULE=="cosine": return max(0.0, 0.5*(1+math.cos(math.pi*prog)))
        return max(0.0, 1-prog)
    return LambdaLR(opt, lr_lambda)

@torch.no_grad()
def _val_macro_f1(model, tok, val_rows):
    """Per-word macro-F1 at 0.5 over the 7 marks (early-stopping signal)."""
    model.eval()
    tp=np.zeros(NUM_MARKS); fp=np.zeros(NUM_MARKS); fn=np.zeros(NUM_MARKS)
    for r in val_rows:
        words=words_of_raw(r["text"]); n=len(words)
        _,Y=gold_to_labels(r["final_text"])
        acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
        for wslice,_,start in chunk_words(words,None):
            enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
            wid=enc.word_ids()
            with torch.autocast("cuda", enabled=(USE_AMP and device=="cuda")):
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits.float()).cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=start+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        pred=((acc/cnt)>=0.5).astype(int)
        tp+=((pred==1)&(Y==1)).sum(0); fp+=((pred==1)&(Y==0)).sum(0); fn+=((pred==0)&(Y==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0,2*prec*rec/np.clip(prec+rec,1e-9,None),0.0)
    return float(f1.mean())

def train_one(model_name, train_rows, val_rows, hp):
    """hp: dict with lr, batch, grad_accum, max_epochs, patience, use_llrd, llrd_decay."""
    tok=AutoTokenizer.from_pretrained(model_name)
    pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tr=PunctDataset(train_rows,tok,has_labels=True)
    dl=DataLoader(tr,batch_size=hp["batch"],shuffle=True,num_workers=2,pin_memory=True,
                  collate_fn=partial(collate,pad_id=pad_id))
    model=PunctModel(model_name).to(device)
    if hp.get("grad_ckpt", False):
        model.backbone.gradient_checkpointing_enable()
    pw=compute_pos_weight(train_rows).to(device)
    opt=build_optimizer(model, hp["lr"], WEIGHT_DECAY,
                        use_llrd=hp.get("use_llrd",False), llrd_decay=hp.get("llrd_decay",0.95))
    accum=max(1,int(hp.get("grad_accum",1)))
    steps_per_epoch=math.ceil(len(dl)/accum)
    total=steps_per_epoch*hp["max_epochs"]
    sched=build_scheduler(opt, total, WARMUP_FRAC, hp["lr"])
    scaler=torch.amp.GradScaler("cuda", enabled=(USE_AMP and device=="cuda"))

    best_f1=-1.0; best_state=None; patience=0
    for ep in range(hp["max_epochs"]):
        model.train(); run=0.0; opt.zero_grad(set_to_none=True)
        for it,(input_ids,attn,active,labels,_) in enumerate(dl):
            input_ids,attn=input_ids.to(device,non_blocking=True),attn.to(device,non_blocking=True)
            active,labels=active.to(device,non_blocking=True),labels.to(device,non_blocking=True)
            with torch.autocast("cuda", enabled=(USE_AMP and device=="cuda")):
                loss=focal_bce(model(input_ids,attn),labels,active,pos_weight=pw)
            scaler.scale(loss/accum).backward()
            run+=loss.item()
            if (it+1)%accum==0 or (it+1)==len(dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                try: sched.step()
                except Exception: pass
        vf1=_val_macro_f1(model,tok,val_rows) if len(val_rows) else -1.0
        tag=f"    epoch {ep+1}/{hp['max_epochs']} loss {run/max(1,len(dl)):.4f}"
        if len(val_rows):
            tag+=f" | val macroF1 {vf1:.4f}"
            if vf1>best_f1+1e-4:
                best_f1=vf1; patience=0; tag+="  *"
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            else:
                patience+=1
        print(tag, flush=True)
        if len(val_rows) and patience>=hp["patience"]:
            print(f"    early stop; best val macroF1 {best_f1:.4f}", flush=True); break
    if best_state is not None: model.load_state_dict(best_state)
    return model, tok, best_f1

@torch.no_grad()
def predict_probs(model, tok, rows):
    """-> [(words, P[n_words, NUM_MARKS] float32)] with center-averaged windows."""
    model.eval(); results=[]
    for r in rows:
        words=words_of_raw(r["text"]); n=len(words)
        acc=np.zeros((n,NUM_MARKS),dtype=np.float32); cnt=np.zeros((n,1),dtype=np.float32)+1e-9
        for wslice,_,start in chunk_words(words,None):
            enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
            wid=enc.word_ids()
            with torch.autocast("cuda", enabled=(USE_AMP and device=="cuda")):
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits.float()).cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=start+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        results.append((words,(acc/cnt).astype(np.float32)))
    return results

# =========================================================================
#  6. Checkpoint paths
# =========================================================================
def slug(mn): return mn.split("/")[-1].replace("-","_")
def ckpt_path(mn,k): return os.path.join(SAVE_DIR, f"{slug(mn)}_fold{k}.pt")

def load_fold_model(mn,k):
    model=PunctModel(mn).to(device)
    model.load_state_dict(torch.load(ckpt_path(mn,k), map_location=device))
    model.eval()
    return model, AutoTokenizer.from_pretrained(mn)

# =========================================================================
#  7. Exact per-class threshold search (sweeps every candidate cutoff)
#     For a sorted-by-prob column: precision=tp/k, recall=tp/npos
#     => F1 = 2*tp/(k+npos).  O(n log n) instead of a coarse grid.
# =========================================================================
def tune_thresholds(Yt, Pp, lo=None, hi=None):
    lo = THR_LO if lo is None else lo; hi = THR_HI if hi is None else hi
    best=np.full(NUM_MARKS,0.5,dtype=np.float64)
    for m in range(NUM_MARKS):
        p=Pp[:,m]; y=Yt[:,m]
        npos=float(y.sum())
        if npos==0: continue
        order=np.argsort(-p, kind="stable"); ps=p[order]; ys=y[order]
        tp=np.cumsum(ys); k=np.arange(1,len(ys)+1,dtype=np.float64)
        f1=2*tp/(k+npos)
        valid=(ps>=lo)&(ps<=hi)
        if not valid.any(): continue
        best[m]=float(ps[int(np.argmax(np.where(valid,f1,-1.0)))])
    return best

def per_class_f1(y_true, y_pred):
    tp=((y_pred==1)&(y_true==1)).sum(0); fp=((y_pred==1)&(y_true==0)).sum(0)
    fn=((y_pred==0)&(y_true==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    return np.where((prec+rec)>0, 2*prec*rec/np.clip(prec+rec,1e-9,None), 0.0)


## 3. Worker → `worker.py` (one model, all folds, one GPU)

In [ ]:
%%writefile /kaggle/working/worker.py
# =========================================================================
#  worker.py — trains ONE model (all folds) on ONE GPU, then writes:
#     {SAVE_DIR}/{slug}_fold{k}.pt      fold checkpoints
#     {SAVE_DIR}/oof_{slug}.pkl         full out-of-fold probs (+ per-fold val F1)
#     {SAVE_DIR}/test_probs_{slug}.pkl  fold-bagged test probs (inference done here,
#                                       so the merge cell needs no GPU at all)
#  The GPU is chosen by CUDA_VISIBLE_DEVICES in the launcher, so this script
#  always sees exactly one device -> "cuda:0" for both workers, no conflicts.
# =========================================================================
import argparse, json, os, pickle, gc, time
import numpy as np, pandas as pd, torch

import muraqam_lib as L

ap = argparse.ArgumentParser()
ap.add_argument("--key", required=True, help="key into CFG['MODELS'] (e.g. base / large)")
args = ap.parse_args()

cfg  = L.CFG
mcfg = cfg["MODELS"][args.key]
mn   = mcfg["name"]
hp   = mcfg["hp"]
sl   = L.slug(mn)
t0   = time.time()

print(f"[{args.key}] model={mn}", flush=True)
print(f"[{args.key}] device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}", flush=True)
print(f"[{args.key}] hp={hp}", flush=True)

df = L.load_df()
print(f"[{args.key}] docs={len(df)} | per fold: {df['fold'].value_counts().sort_index().to_dict()}", flush=True)

# test rows (predicted per fold so we never reload checkpoints later)
test_rows, test_ids, id_col = None, None, None
if os.path.exists(L.TEST_CSV):
    test = pd.read_csv(L.TEST_CSV)
    id_col = "id" if "id" in test.columns else test.columns[0]
    test_ids = test[id_col].tolist()
    test_rows = [{"text": t} for t in test["text"].tolist()]
    print(f"[{args.key}] test docs={len(test_rows)}", flush=True)

oof = [None]*len(df)
fold_val_f1 = []
test_sum = None          # running sum over folds -> bagged mean

for k in range(L.N_FOLDS):
    va_idx = np.where(df["fold"].values == k)[0]
    tr_idx = np.where(df["fold"].values != k)[0]
    tr_rows = [df.iloc[i] for i in tr_idx]
    va_rows = [df.iloc[i] for i in va_idx]

    if L.REUSE_SAVED and os.path.exists(L.ckpt_path(mn,k)):
        print(f"[{args.key}] fold {k}: loading saved checkpoint ({len(va_idx)} OOF docs)", flush=True)
        model, tok = L.load_fold_model(mn, k)
        best = float("nan")
    else:
        print(f"[{args.key}] fold {k}: train {len(tr_idx)} docs | OOF {len(va_idx)} docs", flush=True)
        model, tok, best = L.train_one(mn, tr_rows, va_rows, hp)
        torch.save({kk: v.detach().cpu() for kk,v in model.state_dict().items()}, L.ckpt_path(mn,k))
        print(f"[{args.key}] fold {k}: saved -> {L.ckpt_path(mn,k)}", flush=True)
    fold_val_f1.append(best)

    # --- OOF for the held-out fold ---
    res = L.predict_probs(model, tok, va_rows)
    for j,i in enumerate(va_idx): oof[i] = res[j]

    # --- test probs from this fold (bagging) ---
    if test_rows is not None:
        tr = L.predict_probs(model, tok, test_rows)
        if test_sum is None:
            test_sum = [(w, p.copy()) for (w,p) in tr]
        else:
            for t in range(len(tr)): test_sum[t][1][:] += tr[t][1]
        del tr

    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"[{args.key}] fold {k} done | elapsed {(time.time()-t0)/60:.1f} min", flush=True)

# ---------------- save everything ----------------
with open(os.path.join(L.SAVE_DIR, f"oof_{sl}.pkl"), "wb") as f:
    pickle.dump({"model": mn, "key": args.key, "oof": oof,
                 "ids": df["id"].values if "id" in df.columns else np.arange(len(df)),
                 "fold": df["fold"].values, "fold_val_f1": fold_val_f1}, f)
print(f"[{args.key}] saved OOF -> {L.SAVE_DIR}/oof_{sl}.pkl", flush=True)

if test_sum is not None:
    bagged = [(w, (p/L.N_FOLDS).astype(np.float32)) for (w,p) in test_sum]
    with open(os.path.join(L.SAVE_DIR, f"test_probs_{sl}.pkl"), "wb") as f:
        pickle.dump({"model": mn, "key": args.key, "id_col": id_col,
                     "ids": test_ids, "probs": bagged}, f)
    print(f"[{args.key}] saved bagged test probs -> {L.SAVE_DIR}/test_probs_{sl}.pkl", flush=True)

df[["id","fold"]].to_csv(os.path.join(L.SAVE_DIR, "fold_map.csv"), index=False) if "id" in df.columns else None
print(f"[{args.key}] ALL DONE in {(time.time()-t0)/60:.1f} min | fold val F1: {fold_val_f1}", flush=True)


## 4. Launch both T4s

`CUDA_VISIBLE_DEVICES=0` / `=1` gives each process exactly one card (each sees it as `cuda:0`,
so there is no device-index bookkeeping and no NCCL). `nohup` + log redirection means the bash
cell returns immediately and the jobs survive it; PIDs go to `logs/*.pid` for the monitor.

In [ ]:
%%bash
cd /kaggle/working
mkdir -p logs muraqam_cv
nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

# fresh logs (checkpoints are NOT touched -> REUSE_SAVED still resumes)
: > logs/base.log ; : > logs/large.log

CUDA_VISIBLE_DEVICES=0 nohup python -u worker.py --key base  > logs/base.log  2>&1 &
echo $! > logs/base.pid
CUDA_VISIBLE_DEVICES=1 nohup python -u worker.py --key large > logs/large.log 2>&1 &
echo $! > logs/large.pid

sleep 3
echo "launched -> base pid $(cat logs/base.pid) on GPU0 | large pid $(cat logs/large.pid) on GPU1"

### 4b. Monitor (interleaves both logs, blocks until both finish)

Run this right after the launcher. If it dies the workers keep going — just re-run this cell.
`nvidia-smi` every few minutes tells you if `large` is close to OOM; if it OOMs, drop
`batch` to 2 / `grad_accum` to 4, or flip `grad_ckpt=True` for it, and relaunch (finished
folds are reloaded from disk, not retrained).

In [ ]:
import os, time, subprocess

KEYS  = ["base", "large"]
LOGS  = {k: f"/kaggle/working/logs/{k}.log" for k in KEYS}
PIDS  = {k: int(open(f"/kaggle/working/logs/{k}.pid").read().strip()) for k in KEYS}
pos   = {k: 0 for k in KEYS}
alive = lambda pid: os.path.exists(f"/proc/{pid}")

def drain():
    for k in KEYS:
        if not os.path.exists(LOGS[k]): continue
        with open(LOGS[k], "r") as f:
            f.seek(pos[k]); new = f.read(); pos[k] = f.tell()
        for line in new.splitlines():
            if line.strip(): print(f"{line}", flush=True)

t0 = time.time(); last_smi = 0
while True:
    drain()
    if time.time() - last_smi > 300:
        smi = subprocess.run(["nvidia-smi","--query-gpu=index,utilization.gpu,memory.used",
                              "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        print(f"--- {(time.time()-t0)/60:6.1f} min | {smi.replace(chr(10),'  ||  ')}", flush=True)
        last_smi = time.time()
    if not any(alive(p) for p in PIDS.values()):
        break
    time.sleep(15)
drain()

for k in KEYS:
    ok = "ALL DONE" in open(LOGS[k]).read()
    print(f"[{k}] {'finished OK' if ok else 'DID NOT FINISH -> check logs/'+k+'.log (tail below)'}")
    if not ok:
        print("".join(open(LOGS[k]).readlines()[-25:]))
print(f"total wall clock: {(time.time()-t0)/60:.1f} min")

## 5. Blend weight + thresholds on the pooled OOF

Both OOF sets cover 100% of train and come from folds that never saw the doc, so the base↔large
blend weight can be fit here honestly instead of guessed (the old `MODEL_WEIGHTS = [1.3, 1.0, 2.0]`
was hand-set). For each candidate weight the thresholds are re-tuned — weight and thresholds
interact, tuning them separately leaves F1 on the table. Costs zero training.

In [ ]:
import sys, pickle, json
sys.path.insert(0, "/kaggle/working")
import numpy as np, pandas as pd
import muraqam_lib as L

df   = L.load_df()
KEYS = ["base", "large"]
oofs = {}
for k in KEYS:
    sl = L.slug(CFG["MODELS"][k]["name"])
    with open(f"{SAVE_DIR}/oof_{sl}.pkl", "rb") as f: oofs[k] = pickle.load(f)
    print(f"[{k}] {oofs[k]['model']} | per-fold val macroF1: "
          f"{[round(x,4) if x==x else None for x in oofs[k]['fold_val_f1']]}")

# pooled word-level gold + per-model probs (same word order for every model)
Y = np.concatenate([L.gold_to_labels(df.iloc[i]["final_text"])[1] for i in range(len(df))])
P = {k: np.concatenate([oofs[k]["oof"][i][1] for i in range(len(df))]).astype(np.float32) for k in KEYS}
print("pooled words:", Y.shape[0])

# --- solo scores (for the record) ---
for k in KEYS:
    thr = L.tune_thresholds(Y, P[k])
    f1  = L.per_class_f1(Y, (P[k] >= thr[None,:]).astype(np.float32))
    print(f"  solo {k:5s} OOF macroF1 = {f1.mean():.4f}")

# --- weight sweep (w = weight on LARGE) ---
best_w, best_f1, best_thr = 0.5, -1.0, None
for w in np.linspace(0.0, 1.0, 21):
    Pw  = (1.0-w)*P["base"] + w*P["large"]
    thr = L.tune_thresholds(Y, Pw)
    f1  = L.per_class_f1(Y, (Pw >= thr[None,:]).astype(np.float32)).mean()
    print(f"  w_large={w:.2f}  OOF macroF1={f1:.4f}")
    if f1 > best_f1: best_w, best_f1, best_thr = float(w), float(f1), thr

print(f"\n>>> best blend: w_large={best_w:.2f}  w_base={1-best_w:.2f}  OOF macroF1={best_f1:.4f}")
print("thresholds:", {MARKS[i]: round(float(best_thr[i]),3) for i in range(NUM_MARKS)})

np.save(f"{SAVE_DIR}/thresholds.npy", best_thr)
with open(f"{SAVE_DIR}/blend.json", "w") as f:
    json.dump({"w_large": best_w, "w_base": 1-best_w,
               "thresholds": [float(x) for x in best_thr],
               "oof_macro_f1": best_f1,
               "models": {k: CFG["MODELS"][k]["name"] for k in KEYS}}, f, indent=2)
print("saved ->", f"{SAVE_DIR}/thresholds.npy", "+", f"{SAVE_DIR}/blend.json")

# --- per-class breakdown ---
Pw   = (1.0-best_w)*P["base"] + best_w*P["large"]
f1pc = L.per_class_f1(Y, (Pw >= best_thr[None,:]).astype(np.float32))
sup  = Y.sum(0)
for i,m in enumerate(MARKS):
    print(f"   {m}: F1={f1pc[i]:.3f}  thr={best_thr[i]:.2f}  support={int(sup[i])}")

# --- exact host metric on the pooled OOF (round-trips through reconstruct) ---
oof_texts = []
off = 0
for i in range(len(df)):
    words = oofs[KEYS[0]]["oof"][i][0]; n = len(words)
    oof_texts.append(L.reconstruct(words, L.probs_to_multihot(words, Pw[off:off+n], best_thr)))
    off += n
val_df = pd.DataFrame({"id": range(len(df)), "text": df["text"], "final_text": df["final_text"]})
sub_df = pd.DataFrame({"id": range(len(df)), "final_text": oof_texts})
print(f"\n>>> OOF HOST macro-F1 ({N_FOLDS}-fold, full train): "
      f"{score(val_df.copy(), sub_df.copy(), 'id'):.4f} <<<")

## 6. Submission (CPU only — test probs were already bagged on the GPUs)

In [ ]:
import pickle, numpy as np, pandas as pd, os, json

paths = {k: f"{SAVE_DIR}/test_probs_{L.slug(CFG['MODELS'][k]['name'])}.pkl" for k in KEYS}
if all(os.path.exists(p) for p in paths.values()):
    tp = {}
    for k,p in paths.items():
        with open(p, "rb") as f: tp[k] = pickle.load(f)
    assert tp["base"]["ids"] == tp["large"]["ids"], "test row order differs between workers"

    blend = json.load(open(f"{SAVE_DIR}/blend.json"))
    thr   = np.array(blend["thresholds"], dtype=np.float32)
    wl    = blend["w_large"]
    print(f"blend: base {1-wl:.2f} / large {wl:.2f} | thresholds "
          f"{ {MARKS[i]: round(float(thr[i]),2) for i in range(NUM_MARKS)} }")

    pred_texts = []
    for t in range(len(tp["base"]["probs"])):
        words = tp["base"]["probs"][t][0]
        P = (1-wl)*tp["base"]["probs"][t][1] + wl*tp["large"]["probs"][t][1]
        pred_texts.append(L.reconstruct(words, L.probs_to_multihot(words, P, thr)))

    id_col = tp["base"]["id_col"]
    submission = pd.DataFrame({id_col: tp["base"]["ids"], "final_text": pred_texts})
    submission.to_csv("submission.csv", index=False)
    print("wrote submission.csv", submission.shape)
    print(submission.head(2).to_string())

    # word-count sanity: any preprocessing that changes word count = silent scoring bug
    test = pd.read_csv(CFG["TEST_CSV"])
    bad = sum(len(str(a).strip().split()) != len(str(b).split())
              for a,b in zip(test["text"], submission["final_text"]))
    print("word-count mismatches vs raw test (want 0):", bad)
else:
    print("test probs missing ->", {k:p for k,p in paths.items() if not os.path.exists(p)})
    print("folds + OOF + thresholds are still saved in", SAVE_DIR)